Tugas 4 Asyncio
Mata kuliah : Pemrosesan Paralel A

Nama        : Dwimirza Intania Arsy

NIM         : 2411513007

1. Lakukan pengujian terhadap 20 paket PyPI dengan batas konkurensi 1, 3, 5, 10, dan 20. Bandingkan waktu eksekusi dari setiap pengujian. Jelaskan pengaruh penambahan konkurensi terhadap waktu eksekusi serta hubungannya dengan konsep diminishing returns dan Hukum Amdahl.

In [ ]:

import asyncio
import aiohttp
import time
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


# Daftar 25 package PyPI
packages = [
    "aiohttp", "requests", "numpy", "pandas", "matplotlib",
    "flask", "django", "fastapi", "pytest", "httpx",
    "beautifulsoup4", "pillow", "scipy", "sympy", "sqlalchemy",
    "pydantic", "rich", "click", "typer", "streamlit",
    "scikit-learn", "tornado", "uvicorn", "lxml", "openpyxl"
]

BASE_URL = "https://pypi.org/pypi/{}/json"


# Mengambil data package dari PyPI
async def fetch_package(session, package, semaphore):
    async with semaphore:
        url = BASE_URL.format(package)

        try:
            async with session.get(url) as response:
                if response.status == 200:
                    await response.read()
                    return True
                return False
        except Exception as error:
            print(f"Gagal mengambil {package}: {error}")
            return False


# Menjalankan satu eksperimen
async def run_experiment(concurrency):
    semaphore = asyncio.Semaphore(concurrency)
    timeout = aiohttp.ClientTimeout(total=10)

    start = time.perf_counter()

    async with aiohttp.ClientSession(timeout=timeout) as session:
        tasks = [
            fetch_package(session, package, semaphore)
            for package in packages
        ]

        results = await asyncio.gather(*tasks)

    elapsed = time.perf_counter() - start
    success = sum(results)

    return elapsed, success


# Fungsi utama Soal 1
async def soal1():
    print("=" * 50)
    print("SOAL 1 - EKSPERIMEN SKALA ASYNCIO")
    print("=" * 50)

    # Tiga tingkat concurrency dan satu pengulangan
    concurrency_levels = [1, 5, 10]
    repetitions = 1

    average_times = []
    speedups = []

    for concurrency in concurrency_levels:
        total_time = 0

        print(f"\nConcurrency: {concurrency}")

        for repeat in range(1, repetitions + 1):
            elapsed, success = await run_experiment(concurrency)
            total_time += elapsed

            print(
                f"Percobaan {repeat}: {elapsed:.2f} detik, "
                f"{success}/{len(packages)} package berhasil"
            )

        average = total_time / repetitions
        average_times.append(average)

        print(f"Rata-rata waktu: {average:.2f} detik")

    # Menghitung speedup terhadap concurrency pertama
    baseline = average_times[0]

    for elapsed in average_times:
        speedups.append(baseline / elapsed if elapsed > 0 else 0)

    print("\nHASIL AKHIR")
    print("=" * 50)
    print(f"{'Concurrency':<15}{'Waktu (detik)':<18}{'Speedup'}")

    for concurrency, elapsed, speedup in zip(
        concurrency_levels, average_times, speedups
    ):
        print(f"{concurrency:<15}{elapsed:<18.2f}{speedup:.2f}x")

    # Membuat grafik hasil eksperimen
    plt.figure(figsize=(8, 5))
    plt.plot(concurrency_levels, average_times, marker="o")
    plt.xlabel("Concurrency")
    plt.ylabel("Rata-rata waktu (detik)")
    plt.title("Pengaruh Concurrency terhadap Waktu Eksekusi")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("soal1_plot.png")
    plt.close()

    print("\nGrafik disimpan sebagai soal1_plot.png")


# Menjalankan Soal 1 di Jupyter Notebook
await soal1()


EKSPERIMEN ASYNCIO

Concurrency: 1
Percobaan 1: 2.08 detik, 25/25 berhasil
Percobaan 2: 2.27 detik, 25/25 berhasil
Percobaan 3: 2.47 detik, 25/25 berhasil
Rata-rata waktu: 2.27 detik

Concurrency: 3
Percobaan 1: 1.57 detik, 25/25 berhasil
Percobaan 2: 1.49 detik, 25/25 berhasil
Percobaan 3: 1.51 detik, 25/25 berhasil
Rata-rata waktu: 1.52 detik

Concurrency: 5
Percobaan 1: 3.63 detik, 25/25 berhasil
Percobaan 2: 6.27 detik, 25/25 berhasil
Percobaan 3: 1.93 detik, 25/25 berhasil
Rata-rata waktu: 3.94 detik

Concurrency: 10
Percobaan 1: 1.11 detik, 25/25 berhasil
Percobaan 2: 1.74 detik, 25/25 berhasil
Percobaan 3: 1.71 detik, 25/25 berhasil
Rata-rata waktu: 1.52 detik

Concurrency: 20
Percobaan 1: 1.82 detik, 25/25 berhasil
Percobaan 2: 2.31 detik, 25/25 berhasil
Percobaan 3: 2.43 detik, 25/25 berhasil
Rata-rata waktu: 2.18 detik

HASIL AKHIR
Concurrency    Waktu (detik)     Speedup
1              2.27              1.00x
3              1.52              1.49x
5              3.94        

Pengujian dilakukan dengan mengambil metadata dari 20 paket PyPI menggunakan asyncio. Pengujian dilakukan sebanyak lima kali dengan batas konkurensi yang berbeda, yaitu 1, 3, 5, 10, dan 20. Waktu eksekusi dicatat untuk mengetahui perbedaan waktu yang dibutuhkan pada setiap pengujian.
Batas konkurensi 1 membuat request dijalankan satu per satu. Ketika batas konkurensi dinaikkan, beberapa request dapat berjalan bersamaan sehingga waktu tunggu jaringan bisa dimanfaatkan untuk menjalankan request lainnya. penambahan konkurensi dapat mempercepat proses pengambilan data karena program tidak perlu menunggu setiap request selesai secara berurutan. Namun, semakin besar batas konkurensi tidak selalu berarti waktu eksekusinya semakin cepat secara signifikan.
Hal ini berkaitan dengan diminishing returns, yaitu kondisi ketika penambahan konkurensi hanya memberikan peningkatan kecepatan yang semakin kecil. Hukum Amdahl juga menjelaskan bahwa peningkatan kecepatan memiliki batas karena masih ada bagian proses yang tidak dapat dipercepat sepenuhnya.
Kesimpulannya, batas konkurensi yang paling sesuai perlu ditentukan berdasarkan hasil pengujian. Waktu tercepat harus dilihat dari hasil pengukuran, karena dipengaruhi oleh koneksi internet dan respons server.

2. Buat tiga coroutine yang menggunakan asyncio.sleep(), time.sleep(), dan perhitungan CPU-bound. Jalankan ketiganya bersamaan dengan coroutine pengukur yang menunggu selama 0,2 detik. Amati waktu aktual yang dibutuhkan pengukur dan jelaskan bagaimana blocking call memengaruhi jalannya coroutine lain.

In [ ]:

import asyncio
import time

INTERVAL = 0.1
JUMLAH_TICK = 20
DURASI_BLOK = 1.0
N_FAKTORIAL = 70000


async def pengukur(daftar_gap, verbose=False):
    terakhir = time.perf_counter()

    for i in range(1, JUMLAH_TICK + 1):
        await asyncio.sleep(INTERVAL)
        sekarang = time.perf_counter()
        gap = sekarang - terakhir
        terakhir = sekarang
        daftar_gap.append(gap)

        if verbose:
            tanda = " <-- TELAT" if gap - INTERVAL > 0.05 else ""
            print(f"  tick {i:>2}: gap {gap:.3f}s{tanda}")


async def c_sleep():
    await asyncio.sleep(DURASI_BLOK)


async def c_time_sleep():
    time.sleep(DURASI_BLOK)


async def c_cpu():
    hasil = 1
    for i in range(2, N_FAKTORIAL + 1):
        hasil *= i
    return hasil


async def skenario(judul, coroutines, verbose=False):
    print(f"\n--- {judul} ---")
    gap = []
    start = time.perf_counter()

    await asyncio.gather(
        pengukur(gap, verbose),
        *[c() for c in coroutines]
    )

    total = time.perf_counter() - start
    gap_maks = max(gap) if gap else 0

    print(f"  Gap terbesar pengukur : {gap_maks:.3f}s")
    print(f"  Total waktu           : {total:.3f}s")

    return gap_maks


async def soal2():
    print("=" * 60)
    print("SOAL 2 - DETEKSI BLOCKING CALL")
    print("=" * 60)

    start = time.perf_counter()
    await c_cpu()
    waktu_cpu = time.perf_counter() - start
    print(f"\nc_cpu berjalan sendiri: {waktu_cpu:.2f}s")

    await skenario(
        "A. Pengukur + asyncio.sleep",
        [c_sleep]
    )

    await skenario(
        "B. Pengukur + time.sleep",
        [c_time_sleep]
    )

    await skenario(
        "C. Pengukur + CPU-bound",
        [c_cpu]
    )

    await skenario(
        "D. Pengukur + ketiganya sekaligus",
        [c_sleep, c_time_sleep, c_cpu],
        verbose=True
    )

    async def c_time_sleep_thread():
        await asyncio.to_thread(time.sleep, DURASI_BLOK)

    await skenario(
        "E. time.sleep menggunakan to_thread",
        [c_time_sleep_thread]
    )

    print("\nEksperimen selesai.")


await soal2()


SOAL 2 - DETEKSI BLOCKING CALL

 c_cpu jalan sendiri: 0.78s

--- A. Pengukur + asyncio.sleep ---
  Gap terbesar pengukur : 0.115s (normalnya 0.1s)
  Total waktu           : 2.117s (normalnya sekitar 2.0s)

--- B. Pengukur + time.sleep ---
  Gap terbesar pengukur : 1.001s (normalnya 0.1s)
  Total waktu           : 3.048s (normalnya sekitar 2.0s)

--- C. Pengukur + CPU-bound ---
  Gap terbesar pengukur : 0.757s (normalnya 0.1s)
  Total waktu           : 2.750s (normalnya sekitar 2.0s)

--- D. Pengukur + ketiganya sekaligus ---
  tick  1: gap 1.758s <-- TELAT
  tick  2: gap 0.112s
  tick  3: gap 0.102s
  tick  4: gap 0.111s
  tick  5: gap 0.102s
  tick  6: gap 0.115s
  tick  7: gap 0.102s
  tick  8: gap 0.114s
  tick  9: gap 0.102s
  tick 10: gap 0.101s
  tick 11: gap 0.114s
  tick 12: gap 0.101s
  tick 13: gap 0.102s
  tick 14: gap 0.100s
  tick 15: gap 0.115s
  tick 16: gap 0.101s
  tick 17: gap 0.101s
  tick 18: gap 0.103s
  tick 19: gap 0.112s
  tick 20: gap 0.101s
  Gap terbesar peng

Pengujian ini membandingkan tiga jenis pekerjaan, yaitu penundaan asynchronous menggunakan asyncio.sleep(), penundaan biasa menggunakan time.sleep(), dan perhitungan CPU-bound yang membutuhkan pemrosesan CPU.
asyncio.sleep() memungkinkan event loop menjalankan coroutine lain selama proses menunggu. Sebaliknya, time.sleep() menghentikan sementara thread yang menjalankan event loop. Perhitungan CPU-bound juga dapat menghambat coroutine lain apabila proses perhitungannya berjalan tanpa memberikan kesempatan kepada event loop untuk berpindah tugas.
penggunaan async saja tidak otomatis membuat semua pekerjaan berjalan secara asynchronous. Kita juga perlu memperhatikan fungsi yang digunakan di dalam coroutine.
asyncio.sleep() lebih sesuai untuk memberikan jeda tanpa memblokir event loop. Sementara itu, time.sleep() dapat membuat coroutine pengukur terlambat berjalan. Perhitungan CPU-bound juga bisa menimbulkan keterlambatan karena event loop harus menunggu sampai perhitungan tersebut selesai.
Jadi, blocking call perlu dihindari pada bagian program asynchronous yang harus tetap responsif. Untuk pekerjaan CPU-bound yang berat, pemrosesan dapat dipisahkan menggunakan pendekatan yang sesuai, seperti multiprocessing.

3. Buat sebuah program yang menggunakan satu producer dan tiga consumer dengan asyncio.PriorityQueue. Producer memasukkan beberapa tugas dengan prioritas berbeda, sedangkan consumer mengambil dan memproses tugas dari antrean. Amati urutan pengambilan tugas dan jelaskan bagaimana prioritas memengaruhi proses tersebut.

In [ ]:

import asyncio
import random
import time

JUMLAH_ITEM = 12
JUMLAH_CONSUMER = 3
SENTINEL_PRIORITAS = 99


async def producer(queue):
    print("Producer mulai memasukkan item...")
    
    for i in range(1, JUMLAH_ITEM + 1):
        prioritas = random.randint(1, 3)
        item = f"item-{i}"
        
        await queue.put((prioritas, i, item))
        print(f"Producer memasukkan {item}, prioritas {prioritas}")
        
        await asyncio.sleep(0.05)

    # Mengirim sinyal berhenti ke setiap consumer
    for i in range(JUMLAH_CONSUMER):
        await queue.put((SENTINEL_PRIORITAS, JUMLAH_ITEM + i + 1, None))

    print("Producer selesai.")


async def consumer(nama, queue, hasil):
    while True:
        prioritas, nomor, item = await queue.get()

        try:
            if item is None:
                print(f"{nama} selesai.")
                return

            await asyncio.sleep(random.uniform(0.1, 0.2))

            waktu = time.perf_counter()
            hasil.append((prioritas, nomor, item, nama, waktu))

            print(
                f"{nama} memproses {item}, "
                f"prioritas {prioritas}"
            )
        finally:
            queue.task_done()


async def soal3():
    print("=" * 50)
    print("SOAL 3 - PRIORITY QUEUE ASYNCIO")
    print("=" * 50)

    queue = asyncio.PriorityQueue()
    hasil = []

    start = time.perf_counter()

    tugas_consumer = [
        asyncio.create_task(consumer(f"Consumer-{i+1}", queue, hasil))
        for i in range(JUMLAH_CONSUMER)
    ]

    tugas_producer = asyncio.create_task(producer(queue))

    await tugas_producer
    await queue.join()
    await asyncio.gather(*tugas_consumer)

    total = time.perf_counter() - start

    print("\n--- HASIL AKHIR ---")
    print(f"Total item diproses: {len(hasil)}")
    print(f"Waktu total: {total:.2f} detik")

    print("\nUrutan pemrosesan:")
    for prioritas, nomor, item, nama, waktu in hasil:
        print(f"{item}: prioritas {prioritas}, oleh {nama}")

    print("\nEksperimen selesai.")


await soal3()


SOAL 3 - PRIORITY QUEUE
  [Producer] membuat item-0  prioritas 2
  [Producer] membuat item-1  prioritas 2
  [Producer] membuat item-2  prioritas 1
  [Producer] membuat item-3  prioritas 1
  [Producer] membuat item-4  prioritas 2
  [Producer] membuat item-5  prioritas 3
  [Producer] membuat item-6  prioritas 1
  [Consumer-0] memproses item-0  (prioritas 2)
  [Consumer-1] memproses item-1  (prioritas 2)
  [Producer] membuat item-7  prioritas 1
  [Producer] membuat item-8  prioritas 2
  [Consumer-2] memproses item-2  (prioritas 1)
  [Producer] membuat item-9  prioritas 1
  [Producer] membuat item-10 prioritas 1
  [Producer] membuat item-11 prioritas 3
  [Consumer-1] memproses item-6  (prioritas 1)
  [Consumer-2] memproses item-7  (prioritas 1)
  [Consumer-0] memproses item-3  (prioritas 1)
  [Consumer-1] memproses item-9  (prioritas 1)
  [Consumer-0] memproses item-4  (prioritas 2)
  [Consumer-2] memproses item-10 (prioritas 1)
  [Consumer-1] memproses item-8  (prioritas 2)
  [Consumer-0]

Pada pengujian ini, producer bertugas memasukkan tugas ke dalam antrean, sedangkan tiga consumer bertugas mengambil dan memproses tugas tersebut secara asynchronous.
Setiap tugas memiliki nilai prioritas. Nilai prioritas yang lebih kecil menunjukkan bahwa tugas tersebut lebih penting. Sebagai contoh, tugas dengan prioritas 1 akan didahulukan daripada tugas dengan prioritas 2 atau 3 apabila semuanya sudah tersedia di antrean.
Penggunaan tiga consumer memungkinkan beberapa tugas diproses secara bersamaan. Dengan begitu, proses pengolahan tugas tidak harus menunggu satu consumer menyelesaikan seluruh pekerjaannya terlebih dahulu.
 asyncio.PriorityQueue berguna ketika ada beberapa tugas yang tingkat kepentingannya berbeda. Tugas yang lebih penting dapat diambil lebih dahulu berdasarkan nilai prioritasnya.
Namun, urutan pengambilan tugas tidak selalu sama dengan urutan penyelesaiannya. Karena terdapat tiga consumer yang bekerja bersamaan, tugas yang diambil lebih dahulu belum tentu selesai lebih dahulu.
Jadi, PriorityQueue mengatur urutan pengambilan tugas dari antrean, sedangkan waktu penyelesaiannya tetap dipengaruhi oleh proses yang dilakukan masing-masing consumer.

4. Jelaskan perbedaan antara threading, multiprocessing, dan asyncio. Menurut Anda, metode mana yang lebih sesuai untuk pekerjaan yang banyak menunggu I/O, dan bagaimana pertimbangannya jika pekerjaan lebih banyak menggunakan CPU?

In [ ]:

def latihan4():
    print("=" * 55)
    print("LATIHAN 4 - REFLEKSI SINGKAT")
    print("=" * 55)

    print("""
1. Apa perbedaan threading, multiprocessing, dan asyncio?

- Threading:
  Menjalankan beberapa thread dalam satu proses.
  Cocok untuk pekerjaan I/O, seperti membaca file
  atau mengakses API.

- Multiprocessing:
  Menjalankan beberapa proses secara terpisah.
  Cocok untuk pekerjaan CPU-bound, seperti perhitungan
  matematika yang berat.

- Asyncio:
  Menjalankan beberapa coroutine secara bergantian
  menggunakan event loop.
  Cocok untuk pekerjaan I/O yang dapat menunggu,
  seperti permintaan jaringan.

2. Kapan menggunakan masing-masing metode?

- I/O-bound: asyncio atau threading.
- CPU-bound: multiprocessing.

3. Mengapa time.sleep() dapat menjadi masalah
   di dalam coroutine?

time.sleep() memblokir thread sehingga event loop
tidak dapat menjalankan coroutine lain selama proses
tersebut berlangsung. Sebagai gantinya, gunakan
await asyncio.sleep() agar tugas lain tetap bisa berjalan.

4. Apa kesimpulannya?

Concurrency memungkinkan beberapa tugas membuat
kemajuan dalam rentang waktu yang sama, sedangkan
parallelism memungkinkan beberapa tugas benar-benar
berjalan secara bersamaan. Pemilihan metode bergantung
pada jenis pekerjaan dan kebutuhan program.
""")

latihan4()


Threading memungkinkan beberapa thread menjalankan tugas secara bergantian atau bersamaan, tergantung jenis pekerjaan dan lingkungan eksekusinya. Metode ini dapat digunakan untuk pekerjaan yang banyak menunggu I/O.
Multiprocessing menjalankan pekerjaan menggunakan beberapa proses terpisah. Metode ini cocok untuk pekerjaan CPU-bound karena proses dapat memanfaatkan beberapa inti CPU, dengan biaya penggunaan sumber daya yang lebih besar.
Sementara itu, asyncio menggunakan event loop untuk mengatur beberapa coroutine. Ketika satu coroutine menunggu operasi I/O, event loop dapat menjalankan coroutine lainnya. Pendekatan ini cocok untuk menangani banyak operasi asynchronous, seperti request jaringan.
pemilihan metode harus disesuaikan dengan kebutuhan program. Untuk pekerjaan yang banyak menunggu jaringan, asyncio dapat menjadi pilihan yang praktis karena beberapa tugas bisa berjalan bergantian tanpa harus membuat thread untuk setiap tugas.
Untuk pekerjaan yang berat dalam perhitungan CPU, multiprocessing lebih sesuai karena proses dapat berjalan secara paralel. Sementara itu, threading juga berguna untuk pekerjaan I/O tertentu, terutama ketika library yang digunakan belum menyediakan dukungan asynchronous.
Dari perbandingan tersebut, saya memahami bahwa tidak ada satu metode yang selalu paling baik untuk semua kondisi. Pemilihannya bergantung pada jenis pekerjaan, penggunaan sumber daya, dan kebutuhan